In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, re
import numpy as np
import pandas as pd
from math import sqrt

In [ ]:
directory_name = "/content/drive/MyDrive/WiFi Data/Thief Data" # UPDATE if your folder path differs
input_CSI_file = directory_name + "/Thief1_CSI.csv"
annotation_file = directory_name + "/annotation_Thief1.csv"

In [ ]:
column_titles = ["type","role","mac","rssi","rate","sig_mode","mcs","bandwidth","smoothing","not_sounding",
"aggregation","stbc","fec_coding","sgi","noise_floor","ampdu_cnt","channel","secondary_channel",
"local_timestamp","ant","sig_len","rx_state","real_time_set","real_timestamp","len","CSI_DATA"]

orig = pd.read_csv(input_CSI_file, on_bad_lines='skip', low_memory=False)
orig.columns = column_titles
print("raw shape:", orig.shape)

In [ ]:
# Keep only genuine CSI_DATA rows; coerce real_timestamp numeric so any corrupted values become NaN
cleaned = orig[orig["type"] == "CSI_DATA"].copy()
cleaned["real_timestamp"] = pd.to_numeric(cleaned["real_timestamp"], errors="coerce")
cleaned = cleaned.dropna(subset=["real_timestamp"]).sort_values("real_timestamp").reset_index(drop=True)
cleaned["timestamp"] = cleaned["real_timestamp"] - cleaned["real_timestamp"].iloc[0] # zero-based clock
print("cleaned shape:", cleaned.shape)

In [ ]:
import matplotlib.pyplot as plt

bin_edges = np.arange(0, cleaned["timestamp"].max() + 1, 1.0) # 1-second bins
packet_counts, _ = np.histogram(cleaned["timestamp"], bins=bin_edges)

plt.figure(figsize=(14, 5))
plt.bar(bin_edges[:-1], packet_counts, width=1.0)
plt.title("CSI Packet Count Over Time (zero-amplitude subcarriers removed)")
plt.xlabel("Time (seconds since capture start)")
plt.ylabel("Number of packets")
plt.tight_layout()
plt.show()

In [ ]:
ann = pd.read_csv(annotation_file).rename(columns={"label": "current_action"})
ann["timestamp"] = ann["timestamp"].astype(float) - ann["timestamp"].astype(float).iloc[0]

annotation_index = 0
current_activity = ann.iloc[0]["current_action"]
next_activity_time = float(ann.iloc[1]["timestamp"]) if len(ann) > 1 else float("inf")
annotated_data = []
for idx, row in cleaned.iterrows():
    ts = float(row["timestamp"])
    if ts >= next_activity_time:
        annotation_index += 1
        if annotation_index < len(ann):
            current_activity = ann.iloc[annotation_index]["current_action"]
            next_activity_time = float(ann.iloc[annotation_index + 1]["timestamp"]) if annotation_index + 1 < len(ann) else float("inf")
    annotated_data.append({"timestamp": ts, "current_action": current_activity, "segment_id": annotation_index, "CSI_DATA": row["CSI_DATA"]})

annotated_df = pd.DataFrame(annotated_data)
print(annotated_df["current_action"].value_counts())

In [ ]:
output_rows = []
skipped = 0
for _, r in annotated_df.iterrows():
    m = re.findall(r"\[(.*)\]", str(r["CSI_DATA"]))
    if not m:
        skipped += 1; continue
    try:
        csi_raw = [int(x) for x in m[0].split(" ") if x != '']
        if len(csi_raw) != 128:
            skipped += 1; continue
        imaginary, real = [], []
        for i in range(len(csi_raw)):
            (imaginary if i % 2 == 0 else real).append(csi_raw[i])
        amps = np.array([sqrt(imaginary[i]**2 + real[i]**2) for i in range(64)], dtype=np.float64)
    except (ValueError, IndexError):
        skipped += 1; continue
    output_rows.append((r["timestamp"], r["current_action"], r["segment_id"], amps))

print(f"skipped {skipped} malformed rows, kept {len(output_rows)}")

timestamps = np.array([x[0] for x in output_rows])
labels = np.array([x[1] for x in output_rows])
segment_ids = np.array([x[2] for x in output_rows])
amps_matrix = np.stack([x[3] for x in output_rows])

# Data-driven dead-subcarrier detection - verify, don't assume
zero_frac = (amps_matrix < 1e-9).mean(axis=0)
dead_idx = np.where(zero_frac > 0.95)[0]
keep_idx = np.where(zero_frac <= 0.95)[0]
print("dead subcarrier indices (verified from data):", dead_idx.tolist())
print("keeping", len(keep_idx), "genuinely live subcarriers")

subcarriers = amps_matrix[:, keep_idx]
zero_check = (subcarriers < 1e-9).mean(axis=0)
print("max zero-fraction among KEPT columns:", zero_check.max(), "(should be near 0)")

In [ ]:
df = pd.DataFrame({"timestamp": timestamps, "label": labels, "segment_id": segment_ids})
for i in range(subcarriers.shape[1]):
    df[f"sc{i}"] = subcarriers[:, i]
sc_cols = [c for c in df.columns if c.startswith("sc")]

segments = df.groupby("segment_id")
baselines = {}
for seg_id, g in segments:
    if g["label"].iloc[0] == "none":
        n = len(g)
        tail = g.iloc[int(n*0.7):]
        baselines[seg_id] = tail[sc_cols].mean().values

corrected_rows = []
for seg_id, g in segments:
    label = g["label"].iloc[0]
    if label == "none":
        continue
    prev_baseline = baselines.get(seg_id - 1)
    if prev_baseline is None:
        continue
    vals = g[sc_cols].values - prev_baseline
    sub = pd.DataFrame(vals, columns=sc_cols)
    sub["timestamp"] = g["timestamp"].values
    sub["label"] = label
    sub["segment_id"] = seg_id
    corrected_rows.append(sub)

corrected = pd.concat(corrected_rows, ignore_index=True)

specific_classes = ['Door_Handle_Jiggle', 'Door_OpenClose', 'Dragging', 'Prolong_Stillness', 'Crouch_Walking']
corrected = corrected[corrected["label"].isin(specific_classes)].reset_index(drop=True)
print(corrected["label"].value_counts())

In [ ]:
df2 = corrected.sort_values("timestamp").reset_index(drop=True)

for _ in range(2):
    for c in sc_cols:
        df2[c] = df2[c].rolling(window=5, min_periods=1, center=True).mean()

for c in sc_cols:
    mu, sd = df2[c].mean(), df2[c].std() + 1e-8
    df2[c] = (df2[c] - mu) / sd

df2 = df2.dropna().reset_index(drop=True)
print(df2.shape)

In [ ]:
DROP_PROLONG_STILLNESS = False # set False to keep all 5 classes

if DROP_PROLONG_STILLNESS:
    df2 = df2[df2["label"] != "Prolong_Stillness"].reset_index(drop=True)
    print("Dropped Prolong_Stillness — 4-class run")
else:
    print("Keeping all 5 classes")
print(df2["label"].value_counts())

In [ ]:
WIN = 10
X, y, seg_of = [], [], []
for seg_id, g in df2.groupby("segment_id"):
    g = g.reset_index(drop=True)
    feats = g[sc_cols].values
    label = g["label"].iloc[0]
    for i in range(0, len(g) - WIN):
        X.append(feats[i:i+WIN])
        y.append(label)
        seg_of.append(seg_id)

X = np.array(X); y = np.array(y); seg_of = np.array(seg_of)
print("windowed:", X.shape, y.shape)

In [ ]:
from collections import defaultdict

N_TEST_REPS = 2 # explicit rep count per class, per Nafeez — NOT a percentage

seg_label_map = df2.groupby("segment_id")["label"].first()
by_class = defaultdict(list)
for seg, lbl in seg_label_map.items():
    by_class[lbl].append(seg)

test_segs = set()
for cls, segs in by_class.items():
    segs = sorted(segs) # chronological order
    n_reps = len(segs)
    n_test = min(N_TEST_REPS, n_reps - 1) # never leave a class with 0 training reps
    test_segs.update(segs[-n_test:]) # last N_TEST_REPS repetitions, chronologically
    print(f"{cls}: {n_reps} reps total -> {n_reps - n_test} train / {n_test} test")

test_mask = np.isin(seg_of, list(test_segs))
X_train, y_train = X[~test_mask], y[~test_mask]
X_test, y_test = X[test_mask], y[test_mask]
print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("train class balance:", pd.Series(y_train).value_counts().to_dict())
print("test class balance:", pd.Series(y_test).value_counts().to_dict())

In [ ]:
from keras.models import Sequential
from keras.layers import Conv1D, BatchNormalization, Dropout, GlobalAveragePooling1D, Dense
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

classes = sorted(np.unique(y_train))
y_train_oh = pd.get_dummies(y_train)[classes].values
y_test_oh = pd.get_dummies(pd.Categorical(y_test, categories=classes))[classes].values

In [ ]:
model = Sequential()
model.add(Conv1D(32, 3, activation='relu', padding='same', input_shape=(WIN, X_train.shape[2])))
model.add(BatchNormalization())
model.add(Dropout(0.3))
model.add(Conv1D(32, 3, activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(GlobalAveragePooling1D())
model.add(Dropout(0.4))
model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.4))
model.add(Dense(len(classes), activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

In [ ]:
history = model.fit(X_train, y_train_oh, epochs=40, batch_size=128)

In [ ]:
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
pred_labels = np.array(classes)[y_pred_classes]

print("Classification Report:")
print(classification_report(y_test, pred_labels))

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test_oh)
print("Loss:", loss)
print("Accuracy:", accuracy)

In [ ]:
cm = confusion_matrix(y_test, pred_labels, labels=classes)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
cmn = (cm.astype('float') / cm.sum(axis=1)[:, np.newaxis])
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cmn, annot=True, fmt='.2%', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title("Confusion Matrix (Normalized)")
plt.show()

In [ ]:
from pylab import rcParams
rcParams['figure.figsize'] = 10, 4

plt.plot(history.history['accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train'], loc='upper left')
plt.show()

plt.plot(history.history['loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train'], loc='upper left')
plt.show()